In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import datetime

# 16 CPUS / 128 GB Memory
spark = SparkSession.builder \
    .appName("PushshiftRedditEDA") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.executor.memory", "16g") \
    .config("spark.executor.instances", 6) \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Matplotlib created a temporary cache directory at /scratch/asanchez9/job_48453220/matplotlib-zc0zncly because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Spark version: 3.5.0
Spark UI: http://exp-1-16.expanse.sdsc.edu:4040


In [2]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

#Setting the data and Hugging Face cache directories
DATA_DIR = Path(os.environ.get("PUSHSHIFT_DATA_DIR", "data/raw")).expanduser()
HF_HOME = Path(os.environ.get("HF_HOME", "data/hf_cache")).expanduser()

#Create directories
DATA_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)

# Set Hugging Face cache directory path
os.environ["HF_HOME"] = str(HF_HOME)

# Download dataset from Hugging Face to local data directory using specified cache
snapshot_download(
    repo_id="fddemarco/pushshift-reddit",
    repo_type="dataset",
    local_dir=str(DATA_DIR),
    cache_dir=str(HF_HOME),
    token=False,
)

#Print downloaded data directories
print(f"Downloaded dataset to: {DATA_DIR.resolve()}")
print(f"Hugging Face cache: {HF_HOME.resolve()}")

Fetching 220 files:   0%|          | 0/220 [00:00<?, ?it/s]

Downloaded dataset to: /expanse/lustre/projects/uci157/asanchez9/data/raw
Hugging Face cache: /expanse/lustre/projects/uci157/asanchez9/data/hf_cache


In [3]:
files = sorted(DATA_DIR.rglob("*.parquet"))

if not files:
    raise FileNotFoundError(
        f"No parquet files found under {DATA_DIR.resolve()}. "
        "Check that the download completed successfully."
    )

COLS = [
    "author", "created_utc", "id", "num_comments", "score",
    "selftext", "subreddit", "subreddit_id", "title"
]

def read_one(path):
    return (
        spark.read.parquet(str(path))
        .select(
            *[F.col(c) for c in COLS if c != "created_utc"],
            F.col("created_utc").cast("long").alias("created_utc")
        )
    )

dfs = [read_one(f) for f in files]
df = dfs[0]
for d in dfs[1:]:
    df = df.unionByName(d)

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Files loaded: {len(files)}")
print(f"Partitions: {df.rdd.getNumPartitions()}")

Data directory: /expanse/lustre/projects/uci157/asanchez9/data/raw
Files loaded: 218
Partitions: 3339


In [4]:
df.printSchema()

root
 |-- author: string (nullable = true)
 |-- id: string (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- score: long (nullable = true)
 |-- selftext: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- subreddit_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- created_utc: long (nullable = true)

